In [ ]:
import os
import base64
import io
from datetime import datetime
import json
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mne
import yasa
import sys as _sys
# curry shared modules — located next to this notebook
_here = os.path.dirname(os.path.abspath('__file__'))
if _here not in _sys.path:
    _sys.path.insert(0, _here)
from curry_header import read_curry_header
from curry_io import read_curry_signal, load_hypnogram_curry
import ipywidgets as widgets
from IPython.display import display, HTML
from ipyfilechooser import FileChooser
from scipy.stats import kurtosis as scipy_kurtosis, skew as scipy_skew
from scipy.signal import find_peaks, savgol_filter

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

In [ ]:
def odd(x):
    """Return nearest odd integer to x (required by savgol_filter)."""
    n = int(round(x))
    if n % 2 != 0:
        return n
    return n - 1 if abs(x - (n - 1)) <= abs((n + 1) - x) else n + 1


def compute_signal_metrics(sig_uV):
    """Compute all quality metrics for one EEG channel (unfiltered signal, in uV)."""
    cur_std = float(np.std(sig_uV))
    u = np.unique(sig_uV)

    # Flat signal: fraction of consecutive sample pairs with no change.
    # Tolerance = 2x ADC resolution to handle rounding, minimum 0.06 uV.
    if len(u) > 1:
        res = float(np.min(np.abs(np.diff(u))))
        flat_atol = max(2 * res, 0.06)
    else:
        flat_atol = 0.06
    flat_pct = float(np.isclose(np.diff(sig_uV), 0, atol=flat_atol, rtol=0.0).mean()) * 100

    # Histogram + Savitzky-Golay smoothing + peak detection.
    # SG window = 2% of bin count (at least 5) to smooth noise without flattening real peaks.
    # Peak prominence threshold = 25% of the histogram maximum to ignore minor bumps.
    if cur_std < 1e-10 or len(u) < 3:
        histo, edges = np.histogram(sig_uV, bins=10)
        sg_window = 5
    else:
        histo, edges = np.histogram(sig_uV, bins='scott')
        sg_window = max(5, odd(0.02 * len(histo)))
    bin_centers = (edges[:-1] + edges[1:]) / 2
    y_smooth = savgol_filter(histo.astype(float), window_length=sg_window, polyorder=3)
    prom_thresh = 0.25 * float(np.max(y_smooth)) if np.max(y_smooth) > 0 else 0.0
    peaks, _ = find_peaks(y_smooth, prominence=prom_thresh)

    # Extreme histogram bins: fraction of samples in the outermost bins.
    # Proxy for in-range clipping (signal saturating within the declared EDF physical range).
    total = float(np.sum(histo))
    hist_extreme_pct = float((histo[0] + histo[-1]) / total * 100) if total > 0 else 0.0

    # Percentile-based amplitude metrics: robust characterization of the signal tail.
    # p99 / p99.9 of |amplitude| capture sustained artifacts (movement bursts, electrode pops)
    # without being dominated by a single sample like the max would be.
    abs_uV = np.abs(sig_uV)

    # Distribution statistics.
    # Fischer kurtosis: 0 = normal; >0 = heavy tails/spikes; <0 = flat/platykurtic.
    return {
        'mean_uV': float(np.mean(sig_uV)),
        'std_uV': cur_std,
        'kurtosis': float(scipy_kurtosis(sig_uV, fisher=True)),
        'skewness': float(scipy_skew(sig_uV)),
        'p99_abs_uV': float(np.percentile(abs_uV, 99.0)),
        'p999_abs_uV': float(np.percentile(abs_uV, 99.9)),
        'flat_pct': flat_pct,
        'hist_extreme_pct': hist_extreme_pct,
        'n_peaks': int(len(peaks)),
        'histo': histo,
        'edges': edges,
        'bin_centers': bin_centers,
        'y_smooth': y_smooth,
        'peaks': peaks,
    }


def flag_channel(metrics, thresholds):
    """Return list of flag reasons for a channel, or empty list if within all thresholds."""
    reasons = []
    if metrics['flat_pct'] > thresholds['flat_pct']:
        reasons.append(f"flat_pct={metrics['flat_pct']:.1f}% > {thresholds['flat_pct']}%")
    if metrics['n_peaks'] >= thresholds['n_peaks']:
        shape = 'bimodal' if metrics['n_peaks'] == 2 else 'multimodal'
        reasons.append(f"n_peaks={metrics['n_peaks']} ({shape})")
    # Low std detects dead or near-dead channels (normal EEG std >> 5 µV).
    # Kurtosis was removed: EEG PSG has physiologically high kurtosis (spindles, K-complexes)
    # causing too many false positives (baseline Fp1 kurtosis ~466).
    if metrics['std_uV'] < thresholds['std_low']:
        reasons.append(f"std={metrics['std_uV']:.2f} µV < {thresholds['std_low']} µV (dead/flat channel)")
    # Extreme histogram bins proxy for in-range clipping (signal saturating within
    # the declared EDF physical range.
    if metrics['hist_extreme_pct'] > thresholds['hist_extreme_pct']:
        reasons.append(f"hist_extreme={metrics['hist_extreme_pct']:.2f}% > {thresholds['hist_extreme_pct']}%")
    return reasons


def plot_histogram_figure(metrics, ch_name, y_max=None, x_lim=None):
    """Return a matplotlib figure with the signal amplitude distribution."""
    edges = metrics['edges']
    histo = metrics['histo']
    bin_centers = metrics['bin_centers']
    y_smooth = metrics['y_smooth']
    peaks = metrics['peaks']
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(edges[:-1], histo, width=edges[1] - edges[0], align='edge',
           color='#d4e8f5', edgecolor='#7ab4d0')
    ax.plot(bin_centers, y_smooth, color='#6a51a3', linewidth=2,
            label='Savitzky-Golay smooth')
    ax.plot(bin_centers[peaks], y_smooth[peaks], 'r+', markersize=10, markeredgewidth=2,
            label=f'{len(peaks)} peak(s) detected')
    ax.set_xlabel('Amplitude (μV)', fontsize=11)
    ax.set_ylabel('Counts', fontsize=11)
    if y_max is not None:
        ax.set_ylim(0, y_max * 1.05)
    if x_lim is not None:
        ax.set_xlim(-x_lim, x_lim)
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    ax.legend(fontsize=9)
    plt.tight_layout()
    return fig


def plot_psd_figure(sig_uV, sf):
    """Global PSD (Welch, 4 s windows, 50 % overlap) on the unfiltered signal."""
    # Frequency bands for visual annotation (non-overlapping, covers 0.5–30 Hz).
    BANDS = [
        (0.5,  4,  'δ',  '#cce0f5'),
        (4,    8,  'θ',  '#ccf0d8'),
        (8,   12,  'α',  '#fff2cc'),
        (12,  16,  'σ',  '#f5d0f0'),
        (16,  30,  'β',  "#ffbaba"),
    ]
    # n_overlap derived from n_per_seg (not independently from sf) to guarantee
    # n_overlap < n_per_seg regardless of floating-point rounding in sf.
    # n_fft passed explicitly to prevent MNE from defaulting it to 256 and silently
    # capping n_per_seg, which would cause n_overlap > n_per_seg.
    n_per_seg = int(4 * sf)
    n_overlap = n_per_seg // 2
    psds, freqs = mne.time_frequency.psd_array_welch(
        sig_uV[np.newaxis, :], sfreq=sf,
        fmin=0.5, fmax=min(100.0, sf / 2 - 0.5),
        n_per_seg=n_per_seg, n_overlap=n_overlap, n_fft=n_per_seg,
        verbose=False
    )
    psd_db = 10 * np.log10(np.maximum(psds[0], 1e-10))
    fig, ax = plt.subplots(figsize=(7, 3))
    for flo, fhi, label, color in BANDS:
        fhi_clipped = min(fhi, float(freqs[-1]))
        ax.axvspan(flo, fhi_clipped, alpha=0.20, color=color, zorder=0, linewidth=0)
        ax.text((flo + fhi_clipped) / 2, 0.97, label, ha='center', va='top',
                fontsize=8, color='#555', transform=ax.get_xaxis_transform())
    ax.plot(freqs, psd_db, linewidth=1.0, color='#2a6099', zorder=2)
    ax.axvline(50, color='#c0392b', linewidth=0.8, linestyle='--', alpha=0.6, zorder=3)
    ax.text(50, 0.97, '50 Hz', ha='center', va='top', fontsize=7.5,
            color='#c0392b', transform=ax.get_xaxis_transform())
    ax.set_xlabel('Frequency (Hz)', fontsize=11)
    ax.set_ylabel('Power (dB/Hz)', fontsize=11)
    ax.set_xlim(0.5, float(freqs[-1]))
    band_ticks = [t for t in [0.5, 4, 8, 12, 16, 30, 50, 70, 90] if t <= float(freqs[-1])]
    ax.set_xticks(band_ticks)
    ax.set_xticklabels([str(t) for t in band_ticks], fontsize=9)
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    return fig


def plot_timeseries_figure(sig_uV, sf, y_lim):
    """Raw time series (downsampled to ~10 Hz for display), fixed Y-axis."""
    step = max(1, int(sf / 10))
    t_h = np.arange(0, len(sig_uV), step) / sf / 3600
    fig, ax = plt.subplots(figsize=(10, 2))
    ax.plot(t_h, sig_uV[::step], linewidth=0.3, color='#2a6099', rasterized=True)
    ax.set_xlim(0, len(sig_uV) / sf / 3600)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_xlabel('Time (h)', fontsize=11)
    ax.set_ylabel('µV', fontsize=11)
    ax.axhline(0, color='#bbb', linewidth=0.5, linestyle='--')
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    return fig


# --- Overview overlays: all channels superimposed, one colour per channel -----------------
# Feed the report's first section: a raw.plot_psd-style multi-channel view (time series,
# PSD, amplitude distribution) plus an across-channels metric summary. Format-agnostic
# (no EDF physical-bounds dependency), so the Curry twin reuses them unchanged.
def _channel_colors(n):
    """n visually distinct colours (turbo colormap)."""
    cmap = plt.get_cmap('turbo')
    return [cmap(i / max(1, n - 1)) for i in range(n)] if n > 1 else [cmap(0.5)]


def _overlay_legend(ax):
    """Compact per-channel legend placed to the right of the axes."""
    handles, labels = ax.get_legend_handles_labels()
    ncol = 1 if len(labels) <= 16 else 2
    ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1.01, 1.0),
              ncol=ncol, fontsize=6.5, frameon=False, handlelength=1.2, columnspacing=0.8)


def plot_overlay_timeseries(raw, sf, y_lim=None):
    """Butterfly view: every channel's time series overlaid on one uV axis.
    Reads one channel at a time from `raw` (a cheap slice of the preloaded data) so the
    whole multi-channel recording is never held as a second float64 array."""
    ch_names = raw.ch_names
    n_times = raw.n_times
    step = max(1, int(sf / 10))   # ~10 Hz display resolution
    t_h = np.arange(0, n_times, step) / sf / 3600
    colors = _channel_colors(len(ch_names))
    _alpha = max(0.3, min(0.8, 10.0 / max(1, len(ch_names))))  # more channels -> more transparent
    fig, ax = plt.subplots(figsize=(10, 3.4))
    for i, ch in enumerate(ch_names):
        sig = raw.get_data(picks=[i])[0] * 1e6
        ax.plot(t_h, sig[::step], lw=0.3, color=colors[i], alpha=_alpha, label=ch, rasterized=True)
        del sig
    ax.set_xlim(0, n_times / sf / 3600)
    if y_lim is not None:
        ax.set_ylim(-y_lim, y_lim)
    ax.axhline(0, color='#bbb', lw=0.5, ls='--')
    ax.set_xlabel('Time (h)', fontsize=11)
    ax.set_ylabel('µV', fontsize=11)
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    _overlay_legend(ax)
    fig.subplots_adjust(right=0.82)
    return fig


def plot_overlay_distribution(raw, x_lim=None):
    """Amplitude distribution (density) of every channel overlaid as outline curves
    (one channel at a time)."""
    ch_names = raw.ch_names
    colors = _channel_colors(len(ch_names))
    _alpha = max(0.3, min(0.8, 10.0 / max(1, len(ch_names))))  # more channels -> more transparent
    fig, ax = plt.subplots(figsize=(9, 3.6))
    for i, ch in enumerate(ch_names):
        sig = raw.get_data(picks=[i])[0] * 1e6
        bins = 'scott' if float(np.std(sig)) > 1e-10 else 10
        histo, edges = np.histogram(sig, bins=bins)
        width = edges[1] - edges[0]
        total = histo.sum()
        dens = histo / (total * width) if (total > 0 and width > 0) else histo.astype(float)
        centers = (edges[:-1] + edges[1:]) / 2
        ax.plot(centers, dens, lw=1.0, color=colors[i], alpha=_alpha, label=ch)
        del sig
    if x_lim is not None:
        ax.set_xlim(-x_lim, x_lim)
    ax.set_xlabel('Amplitude (µV)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    _overlay_legend(ax)
    fig.subplots_adjust(right=0.82)
    return fig


def plot_overlay_psd(raw, sf):
    """Welch PSD (4 s windows) of every channel overlaid on one axis (one channel at a time)."""
    ch_names = raw.ch_names
    n_per_seg = int(4 * sf)
    n_overlap = n_per_seg // 2
    colors = _channel_colors(len(ch_names))
    _alpha = max(0.3, min(0.8, 10.0 / max(1, len(ch_names))))  # more channels -> more transparent
    fig, ax = plt.subplots(figsize=(9, 3.6))
    freqs = None
    for i, ch in enumerate(ch_names):
        sig = raw.get_data(picks=[i])[0] * 1e6
        psds, freqs = mne.time_frequency.psd_array_welch(
            sig[np.newaxis, :], sfreq=sf, fmin=0.5, fmax=min(100.0, sf / 2 - 0.5),
            n_per_seg=n_per_seg, n_overlap=n_overlap, n_fft=n_per_seg, verbose=False)
        psd_db = 10 * np.log10(np.maximum(psds[0], 1e-10))
        ax.plot(freqs, psd_db, lw=0.9, color=colors[i], alpha=_alpha, label=ch)
        del sig, psds
    if freqs is not None:
        ax.set_xlim(0.5, float(freqs[-1]))
    ax.set_xlabel('Frequency (Hz)', fontsize=11)
    ax.set_ylabel('Power (dB/Hz)', fontsize=11)
    for sp in ['right', 'top']:
        ax.spines[sp].set_visible(False)
    _overlay_legend(ax)
    fig.subplots_adjust(right=0.82)
    return fig


 

def build_avg_metrics_html(ch_metrics, ch_names):
    """Table of per-channel metrics summarised across channels (mean/median/min/max).
    Each metric is guarded on key presence, so bounds_pct (EDF-only) is skipped for
    float-based recordings without any code change."""
    METRICS = [
        ('std_uV', 'Std dev (\u00b5V)'), ('flat_pct', 'Flat signal (%)'),
        ('bounds_pct', 'At EDF bounds (%)'), ('hist_extreme_pct', 'Extreme histogram (%)'),
        ('p99_abs_uV', 'p99 |amplitude| (\u00b5V)'), ('p999_abs_uV', 'p99.9 |amplitude| (\u00b5V)'),
        ('kurtosis', 'Kurtosis (Fischer)'), ('skewness', 'Skewness'), ('n_peaks', 'n_peaks'),
    ]
    body = ''
    for key, label in METRICS:
        vals = [ch_metrics[ch][key] for ch in ch_names if key in ch_metrics[ch]]
        if not vals:
            continue
        arr = np.asarray(vals, dtype=float)
        body += (f'<tr><td style="padding:4px 10px;">{label}</td>'
                 f'<td style="padding:4px 10px;text-align:right;">{arr.mean():.3g}</td>'
                 f'<td style="padding:4px 10px;text-align:right;">{np.median(arr):.3g}</td>'
                 f'<td style="padding:4px 10px;text-align:right;">{arr.min():.3g}</td>'
                 f'<td style="padding:4px 10px;text-align:right;">{arr.max():.3g}</td></tr>')
    return (
        f'<p style="margin:6px 0;">Summary across the {len(ch_names)} plotted channel(s).</p>'
        '<table style="border-collapse:collapse;font-size:0.9em;">'
        '<tr style="background:#eee;">'
        '<th style="padding:4px 10px;text-align:left;">Metric</th>'
        '<th style="padding:4px 10px;text-align:right;">Mean</th>'
        '<th style="padding:4px 10px;text-align:right;">Median</th>'
        '<th style="padding:4px 10px;text-align:right;">Min</th>'
        '<th style="padding:4px 10px;text-align:right;">Max</th></tr>'
        f'{body}</table>'
    )


# Per-metric interpretation shown as a third column in the metrics table.
METRIC_INTERPRETATIONS = {
    'Kurtosis (Fischer)': 'PSG EEG: typically ~100&ndash;300; compare across channels &mdash; a channel significantly lower than the others&nbsp;=&nbsp;suspicious',
    'Skewness': '~0&nbsp;=&nbsp;symmetric; |skew|&nbsp;&gt;&nbsp;1&nbsp;=&nbsp;notable asymmetry',
    'n_peaks (distribution)': '1&nbsp;=&nbsp;unimodal; 2&nbsp;=&nbsp;bimodal (DC drift); &ge;3&nbsp;=&nbsp;coarse quantization',
    'p99 |amplitude|': '99% of samples are below this value&nbsp;; compare across channels&nbsp;',
    'p99.9 |amplitude|': '99.9% of samples are below this value&nbsp;; compare across channels&nbsp;',
}

def _fig_to_b64(fig):
    """Render a matplotlib figure to a base64-encoded PNG for HTML embedding."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return b64


# --- Custom (non-AASM) sleep stages -------------------------------------------
# Projects may intentionally keep labels outside the AASM set (W/N1/N2/N3/R). They are
# declared in config_param/custom_stages.json (written by 3_remap_hypno) and recognised
# here so the hypnospectrogram and per-stage tables include them.
BASE_STAGE_COLORS = {'W': '#969696', 'N1': '#9e9ac8', 'N2': '#807dba', 'N3': '#6a51a3', 'R': '#c994c7'}
# Distinct, non-red palette for custom stages (red is reserved for REM on the hypnogram).
CUSTOM_STAGE_PALETTE = ['#8dd3c7', '#ffffb3', '#bebada', '#80b1d3', '#fdb462', '#b3de69', '#fccde5', '#d9d9d9']


def load_custom_stages(folder):
    """Read config_param/custom_stages.json; return list of kept non-AASM labels ([] if absent/unreadable)."""
    if not folder:
        return []
    path = Path(folder) / 'config_param' / 'custom_stages.json'
    if not path.exists():
        return []
    try:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        stages = data.get('custom_stages', []) if isinstance(data, dict) else []
        return [str(s) for s in stages]
    except Exception:
        return []


def parse_custom_field(text):
    """Parse the comma-separated 'Custom stages' field into a clean, de-duplicated list."""
    seen = []
    for tok in str(text).split(','):
        tok = tok.strip()
        if tok and tok not in seen:
            seen.append(tok)
    return seen


def custom_stage_style(custom_stages):
    """Stage -> y-position, stage -> colour, and axis ticks for AASM + custom stages.
    AASM keeps YASA heights (W top ... N3 bottom); custom stages stack below N3 (-1, -2, ...)
    in declaration order with auto-assigned colours. Returns (stage_y, stage_colors, ytick_pos, ytick_labels)."""
    stage_y = {'W': 4, 'R': 3, 'N1': 2, 'N2': 1, 'N3': 0}
    stage_colors = dict(BASE_STAGE_COLORS)
    for i, cs in enumerate(custom_stages):
        stage_y[cs] = -1 - i
        stage_colors[cs] = CUSTOM_STAGE_PALETTE[i % len(CUSTOM_STAGE_PALETTE)]
    ordered = sorted(stage_y.items(), key=lambda kv: kv[1])
    ytick_pos = [y for _, y in ordered]
    ytick_labels = ['REM' if s == 'R' else s for s, _ in ordered]
    return stage_y, stage_colors, ytick_pos, ytick_labels


def plot_hypnospectrogram(sig_uV, sf, hypno_labels, custom_stages, fmin=0.5, fmax=40, trimperc=2.5):
    """Multitaper spectrogram (YASA core) with a custom hypnogram strip that supports AASM +
    project-specific custom stages. The spectrogram is stage-agnostic (reuses yasa's
    spectrogram_lspopt); only the top hypnogram band is drawn here, so any custom label is
    plottable (YASA's own Hypnogram overlay hard-rejects non-AASM labels). Returns a Figure."""
    from yasa.plotting import spectrogram_lspopt
    from matplotlib.colors import Normalize

    hypno_labels = np.asarray(hypno_labels)
    stage_y, stage_colors, ytick_pos, ytick_labels = custom_stage_style(custom_stages)

    # Multitaper spectrogram (mirrors yasa.plot_spectrogram internals).
    nperseg = int(30 * sf)
    f, t, Sxx = spectrogram_lspopt(sig_uV, sf, nperseg=nperseg, noverlap=0)
    with np.errstate(divide='ignore'):
        Sxx = 10 * np.log10(Sxx)
    good = np.logical_and(f >= fmin, f <= fmax)
    Sxx, f = Sxx[good, :], f[good]
    t = t / 3600  # seconds -> hours
    # Flat / disconnected epochs (whole 30 s columns) have near-zero power -> extreme negative
    # dB; left in, they drag the percentile colour scale down and wash the plot to one colour,
    # so exclude them from vmin/vmax and render them grey ('no signal').
    Sxx = np.where(np.isfinite(Sxx), Sxx, np.nan)
    with np.errstate(invalid='ignore'):
        col_max = np.nanmax(np.where(np.isfinite(Sxx), Sxx, -np.inf), axis=0)
    col_max[~np.isfinite(col_max)] = np.nan
    ref = np.nanmedian(col_max)
    valid = np.isfinite(col_max) & (col_max > ref - 60)   # 60 dB below the median epoch peak
    pix = Sxx[:, valid]
    pix = pix[np.isfinite(pix)]
    if pix.size < 10:        # almost everything dead -> graceful fallback
        pix = Sxx[np.isfinite(Sxx)]
    vmin, vmax = np.percentile(pix, [0 + trimperc, 100 - trimperc])
    norm = Normalize(vmin=vmin, vmax=vmax)
    Sxx[:, ~valid] = np.nan   # dead epochs rendered as the colormap 'bad' colour
    spec_cmap = plt.get_cmap('RdBu_r').copy()
    spec_cmap.set_bad('#d9d9d9')

    fig, (ax0, ax1) = plt.subplots(
        nrows=2, figsize=(10, 4.2),
        gridspec_kw={'height_ratios': [1, 2], 'hspace': 0.1}
    )

    # Hypnogram strip: one y per 30s epoch; gray line, REM red, each custom stage in its colour.
    n_ep = len(hypno_labels)
    ep_hours = np.arange(n_ep + 1) * 30 / 3600
    y = np.array([stage_y.get(s, np.nan) for s in hypno_labels], dtype=float)
    y_step = np.append(y, y[-1] if n_ep else np.nan)
    ax0.step(ep_hours, y_step, where='post', color='k', lw=1.0)
    for s in (['R'] + list(custom_stages)):
        col = 'red' if s == 'R' else stage_colors.get(s)
        for ei in np.where(hypno_labels == s)[0]:
            ax0.plot([ep_hours[ei], ep_hours[ei + 1]], [stage_y[s], stage_y[s]],
                     color=col, lw=2.2, solid_capstyle='butt')
    ax0.set_yticks(ytick_pos)
    ax0.set_yticklabels(ytick_labels, fontsize=8)
    ax0.set_ylim(min(ytick_pos) - 0.8, max(ytick_pos) + 0.8)
    ax0.set_xlim(0, ep_hours[-1] if n_ep else 1)
    ax0.xaxis.set_visible(False)
    for sp in ['top', 'right', 'bottom']:
        ax0.spines[sp].set_visible(False)

    # Spectrogram.
    im = ax1.pcolormesh(t, f, Sxx, norm=norm, cmap=spec_cmap, antialiased=True, shading='auto')
    ax1.set_xlim(0, t.max() if len(t) else 1)
    ax1.set_ylabel('Frequency [Hz]', fontsize=11)
    ax1.set_xlabel('Time [hrs]', fontsize=11)
    ax1.tick_params(labelsize=8)
    return fig


def generate_dataset_overview(df, reports_root, df_stage=None, custom_stages=None):
    """
    Generate dataset_overview.html from a quality_summary DataFrame.

    Global section:
      - stats table (all electrodes pooled)
      - mean/median by sleep stage table (if df_stage provided)
      - n_peaks frequency table by electrode
      - pooled distribution plot (one box per metric, with per-stage inset if df_stage provided)
      - grouped comparison plot (one box per metric per electrode)

    Per-electrode sections:
      - stats table for that electrode
      - distribution plot across participants (scatter colored red=flagged, grey=ok)

    Regenerated each run from the full cumulative quality_summary.tsv.
    """
    KEY_METRICS = [
        "std_uV", "flat_pct", "hist_extreme_pct",
        "p99_abs_uV", "p999_abs_uV",
    ]
    ALL_NUMERIC = [
        "mean_uV", "std_uV", "kurtosis", "skewness",
        "p99_abs_uV", "p999_abs_uV",
        "flat_pct", "hist_extreme_pct",
    ]
    LABELS = {
        "std_uV": "Std dev (µV)",
        "flat_pct": "Flat signal (%)",
        "hist_extreme_pct": "Extreme histogram (%)",
        "p99_abs_uV": "p99 |amplitude| (µV)",
        "p999_abs_uV": "p99.9 |amplitude| (µV)",
        "mean_uV": "Mean (µV)",
        "kurtosis": "Kurtosis (Fischer)",
        "skewness": "Skewness",
    }
    custom_stages = custom_stages or []
    _sy, SS_COLORS, _ytp, _ytl = custom_stage_style(custom_stages)
    STAGE_ORDER = ['W', 'N1', 'N2', 'N3', 'R'] + list(custom_stages)

    channels = sorted(df["channel"].unique())
    n_participants = int(df["file_id"].nunique())
    n_entries = len(df)
    n_flagged = int(df["exclude"].sum()) if "exclude" in df.columns else "?"
    n_flagged_part = int(df.groupby("file_id")["exclude"].any().sum()) if "exclude" in df.columns else "?"

    CSS = (
        "<style>"
        "body{font-family:Arial,sans-serif;max-width:1100px;margin:0 auto;padding:20px;color:#333}"
        "h1{color:#333;border-bottom:2px solid #ccc;padding-bottom:8px}"
        "h2{color:#333;margin-top:32px;border-bottom:1px solid #ddd;padding-bottom:4px}"
        "h3{color:#333;margin-top:16px}"
        "table{border-collapse:collapse;width:100%;font-size:.9em;margin-bottom:20px}"
        "th{background:#888;color:#fff;padding:6px 12px;text-align:left}"
        "td{padding:5px 12px;border-bottom:1px solid #eee}"
        "tr:nth-child(even){background:#f8f8f8}"
        ".flag{color:#c0392b;font-weight:bold}"
        "img{max-width:100%;margin:10px 0}"
        ".footer{color:#888;font-size:.8em;margin-top:40px;"
        "border-top:1px solid #ddd;padding-top:10px}"
        "</style>"
    )

    def _stats_table(data):
        HDR = (
            "<tr><th>Metric</th><th>Mean</th><th>Median</th>"
            "<th>p5</th><th>p25</th><th>p75</th><th>p95</th></tr>"
        )
        rows = []
        for m in ALL_NUMERIC:
            s = data[m].dropna() if m in data.columns else pd.Series(dtype=float)
            if s.empty:
                continue
            rows.append(
                f"<tr><td>{LABELS.get(m, m)}</td>"
                f"<td>{s.mean():.3g}</td><td>{s.median():.3g}</td>"
                f"<td>{s.quantile(0.05):.3g}</td><td>{s.quantile(0.25):.3g}</td>"
                f"<td>{s.quantile(0.75):.3g}</td><td>{s.quantile(0.95):.3g}</td></tr>"
            )
        return f"<table>{HDR}{''.join(rows)}</table>"

    def _stage_table():
        """Mean (median) by sleep stage — metrics as rows, stages as columns."""
        if df_stage is None or df_stage.empty:
            return ""
        STAGE_METRICS = ["mean_uV", "std_uV", "flat_pct", "hist_extreme_pct", "p99_abs_uV", "p999_abs_uV"]
        present_stages = [s for s in STAGE_ORDER if s in df_stage['stage'].values]
        if not present_stages:
            return ""
        hdr_stages = "".join(f"<th>{s}</th>" for s in present_stages)
        HDR = f"<tr><th>Metric</th>{hdr_stages}</tr>"
        rows = []
        for m in STAGE_METRICS:
            if m not in df_stage.columns:
                continue
            cells = []
            for stage in present_stages:
                vals = df_stage[df_stage['stage'] == stage][m].dropna()
                if vals.empty:
                    cells.append("<td>—</td>")
                else:
                    cells.append(f"<td>{vals.mean():.3g} ({vals.median():.3g})</td>")
            rows.append(f"<tr><td>{LABELS.get(m, m)}</td>{''.join(cells)}</tr>")
        if not rows:
            return ""
        return (
            "<h3>Mean (median) by sleep stage — all electrodes pooled</h3>"
            f"<table>{HDR}{''.join(rows)}</table>"
            "<p style='font-size:0.85em;color:#666;margin-top:-12px;'>"
            "Values pooled across all channels and participants for each stage. "
            "Format: mean (median).</p>"
        )

    def _npeaks_table(data):
        if "n_peaks" not in data.columns:
            return ""
        HDR = (
            "<tr><th>Electrode</th><th>1 peak (normal)</th>"
            "<th>2 peaks (DC drift)</th><th>≥3 peaks (quantization)</th></tr>"
        )
        rows = []
        for ch in sorted(data["channel"].unique()):
            s = data[data["channel"] == ch]["n_peaks"]
            c1 = int((s == 1).sum())
            c2 = int((s == 2).sum())
            c3 = int((s >= 3).sum())
            f2 = ' class="flag"' if c2 else ""
            f3 = ' class="flag"' if c3 else ""
            rows.append(
                f"<tr><td>{ch}</td><td>{c1}</td>"
                f"<td{f2}>{c2}</td><td{f3}>{c3}</td></tr>"
            )
        return (
            "<h3>n_peaks distribution by electrode</h3>"
            f"<table>{HDR}{''.join(rows)}</table>"
        )

    def _scatter_with_flag(ax, x_center, vals, flags, jitter_width=0.07):
        """Scatter individual data points colored by flag status.
        Red = channel flagged for this participant; grey = within thresholds.
        showfliers=False must be set on the boxplot to avoid duplicate markers.
        """
        colors = ["#c0392b" if f else "#555555" for f in flags]
        jitter = np.random.uniform(-jitter_width, jitter_width, len(vals))
        for v, j, c in zip(vals, jitter, colors):
            ax.scatter(x_center + j, v, color=c, s=28, alpha=0.75, zorder=3,
                       linewidths=0)

    def _flag_legend(fig):
        """Add a flag/ok legend to the figure."""
        handles = [
            matplotlib.lines.Line2D(
                [0], [0], marker="o", color="w", markerfacecolor="#c0392b",
                markersize=8, label="Flagged",
            ),
            matplotlib.lines.Line2D(
                [0], [0], marker="o", color="w", markerfacecolor="#555555",
                markersize=8, label="Not flagged",
            ),
        ]
        fig.legend(handles=handles, loc="upper right", fontsize=9, framealpha=0.9)

    def _vals_and_flags(data, metric):
        """Return aligned (values, flags) arrays for rows where metric is not NaN."""
        valid = data[metric].notna()
        vals = data.loc[valid, metric].values
        if "exclude" in data.columns:
            flags = data.loc[valid, "exclude"].fillna(False).astype(bool).values
        else:
            flags = np.zeros(len(vals), dtype=bool)
        return vals, flags

    def _pooled_boxplots(data):
        """All electrodes combined — one box per metric.
        Each subplot contains a per-stage inset (upper-right corner) when stage data is available.
        Gives the dataset-wide distribution and the per-stage breakdown in a single compact view.
        """
        # Pre-compute stage info once for all subplots
        present_stages = []
        if df_stage is not None and not df_stage.empty:
            present_stages = [s for s in STAGE_ORDER if s in df_stage['stage'].values]

        ncols = min(3, len(KEY_METRICS))
        nrows = int(np.ceil(len(KEY_METRICS) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
        axes = axes.flatten()
        for ax, m in zip(axes, KEY_METRICS):
            vals, flags = _vals_and_flags(data, m)
            if len(vals) == 0:
                ax.set_visible(False)
                continue
            bp = ax.boxplot(
                [vals], patch_artist=True, showfliers=False,
                medianprops=dict(color="#c0392b", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
            )
            bp["boxes"][0].set_facecolor("#d4e8f5")
            bp["boxes"][0].set_alpha(0.8)
            _scatter_with_flag(ax, x_center=1, vals=vals, flags=flags)
            ax.set_title(LABELS.get(m, m), fontsize=11, fontweight="bold")
            ax.set_xticks([])
            for sp in ["right", "top"]:
                ax.spines[sp].set_visible(False)

            # Per-stage inset (upper-right). The main boxplot is pushed against the
            # y-axis by widening the x-limits to the right, so the box + points and the
            # inset never overlap while the left gap stays minimal. The box sits at x=1
            # (spans 0.75–1.25); lower bound 0.65 keeps it tight to the y-axis, upper
            # bound 2.6 frees the right ~half of the axes for the inset.
            if present_stages and df_stage is not None and m in df_stage.columns:
                ax.set_xlim(0.65, 2.6)
                ax_in = ax.inset_axes([0.42, 0.20, 0.56, 0.74])
                groups_in = [df_stage[df_stage['stage'] == s][m].dropna().values
                             for s in present_stages]
                bp_in = ax_in.boxplot(
                    groups_in, labels=present_stages, patch_artist=True, showfliers=False,
                    medianprops=dict(color='#333', linewidth=1.5),
                    whiskerprops=dict(linewidth=1.0),
                    capprops=dict(linewidth=1.0),
                )
                for patch, stage in zip(bp_in['boxes'], present_stages):
                    patch.set_facecolor(SS_COLORS[stage])
                    patch.set_alpha(0.85)
                # Keep the inset y-axis visible so per-stage absolute values stay readable.
                ax_in.tick_params(axis='x', labelsize=7)
                ax_in.tick_params(axis='y', labelsize=7)
                for sp in ['right', 'top']:
                    ax_in.spines[sp].set_visible(False)
                ax_in.patch.set_facecolor('#fafafa')
                ax_in.patch.set_alpha(0.9)

        for ax in axes[len(KEY_METRICS):]:
            ax.set_visible(False)
        _flag_legend(fig)
        fig.suptitle(
            f"All electrodes pooled — metric distributions (n={len(data)} channel records)",
            fontsize=12, y=1.02,
        )
        plt.tight_layout()
        return f'<img src="data:image/png;base64,{_fig_to_b64(fig)}" style="max-width:100%;">'

    def _grouped_boxplots(data):
        """One subplot per metric; x-axis = electrode — compares electrodes side by side.
        showfliers=False keeps scale anchored to the IQR range (no outlier points
        inflating the axis). Individual values are not shown here; see per-electrode
        sections for participant-level detail.
        """
        ncols = min(3, len(KEY_METRICS))
        nrows = int(np.ceil(len(KEY_METRICS) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
        axes = axes.flatten()
        for ax, m in zip(axes, KEY_METRICS):
            groups = [data[data["channel"] == ch][m].dropna().values for ch in channels]
            bp = ax.boxplot(
                groups, labels=channels, patch_artist=True, showfliers=False,
                medianprops=dict(color="#c0392b", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
            )
            for patch in bp["boxes"]:
                patch.set_facecolor("#d4e8f5")
                patch.set_alpha(0.8)
            ax.set_title(LABELS.get(m, m), fontsize=11, fontweight="bold")
            rot = 30 if len(channels) > 4 else 0
            ax.tick_params(axis="x", labelrotation=rot, labelsize=9)
            for sp in ["right", "top"]:
                ax.spines[sp].set_visible(False)
        for ax in axes[len(KEY_METRICS):]:
            ax.set_visible(False)
        plt.tight_layout()
        return f'<img src="data:image/png;base64,{_fig_to_b64(fig)}" style="max-width:100%;">'

    def _electrode_boxplots(ch_data, ch_name):
        """One subplot per metric; distribution of that metric across participants
        for this electrode. Each dot = one participant.
        Red dot = this channel was flagged for that participant.
        Grey dot = within all thresholds.
        showfliers=False removes matplotlib's default outlier markers so each
        participant appears exactly once as a colored dot.
        """
        ncols = min(3, len(KEY_METRICS))
        nrows = int(np.ceil(len(KEY_METRICS) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
        axes = axes.flatten()
        for ax, m in zip(axes, KEY_METRICS):
            vals, flags = _vals_and_flags(ch_data, m)
            if len(vals) == 0:
                ax.set_visible(False)
                continue
            bp = ax.boxplot(
                [vals], patch_artist=True, showfliers=False,
                medianprops=dict(color="#c0392b", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
            )
            bp["boxes"][0].set_facecolor("#d4e8f5")
            bp["boxes"][0].set_alpha(0.8)
            _scatter_with_flag(ax, x_center=1, vals=vals, flags=flags)
            ax.set_title(LABELS.get(m, m), fontsize=11, fontweight="bold")
            ax.set_xticks([])
            for sp in ["right", "top"]:
                ax.spines[sp].set_visible(False)
        for ax in axes[len(KEY_METRICS):]:
            ax.set_visible(False)
        _flag_legend(fig)
        fig.suptitle(
            f"Electrode {ch_name} — distribution across {len(ch_data)} participants",
            fontsize=12, y=1.02,
        )
        plt.tight_layout()
        return f'<img src="data:image/png;base64,{_fig_to_b64(fig)}" style="max-width:100%;">'

    # Build global section
    global_block = (
        "<h2>All electrodes — pooled statistics</h2>"
        f"<h3>Summary statistics (all channels combined, n={n_entries})</h3>"
        + _stats_table(df)
        + _stage_table()
        + _npeaks_table(df)
        + "<h3>Pooled distribution of each metric (all electrodes combined)</h3>"
        + _pooled_boxplots(df)
        + "<h3>Distributions by electrode (comparison)</h3>"
        + _grouped_boxplots(df)
    )

    # Build per-electrode sections
    electrode_blocks = ""
    for ch in channels:
        ch_data = df[df["channel"] == ch]
        electrode_blocks += (
            f"<h2>Electrode: {ch}</h2>"
            f"<h3>Statistics across {len(ch_data)} participants</h3>"
            + _stats_table(ch_data)
            + _electrode_boxplots(ch_data, ch)
        )

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    html = (
        '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
        "<title>Dataset Quality Overview</title>"
        f"{CSS}</head><body>"
        "<h1>Dataset Quality Overview</h1>"
        f"<p><b>Dataset:</b> {reports_root.parent.name}&nbsp;|&nbsp;"
        f"<b>Participants:</b> {n_participants}&nbsp;|&nbsp;"
        f"<b>Entries (channel × participant):</b> {n_entries}&nbsp;|&nbsp;"
        f"<b>Flagged channels:</b> {n_flagged}&nbsp;|&nbsp;"
        f"<b>Participants with ≥1 flag:</b> {n_flagged_part}</p>"
        + global_block
        + electrode_blocks
        + '<p class="footer">'
        f"Generated by 5_quality_overview_voila.ipynb — {timestamp} — "
        f"Source: quality_summary.tsv ({n_entries} entries, {n_participants} participants)"
        "</p></body></html>"
    )

    (reports_root / "dataset_overview.html").write_text(html, encoding="utf-8")

## Preprocessing &mdash; Phase 1: Quality Overview 

This tool computes **signal quality metrics per channel** for every EDF file in a folder. It produces:
- One **HTML report** per participant (histograms, PSDs, time series, descriptive stats, hypnograms/spectrograms)
- A **`quality_summary.tsv`** with all numeric metrics
- A **`dataset_overview.html`** with dataset-level statistics and distributions per electrode

**Instructions:**
1. Use the first file chooser to select your **data folder**.
2. Select the **remap/reref config JSON** produced by *select&amp;remap_channels_edf*.
3. Adjust the **hypnogram suffix** if needed and the **quality thresholds**.
4. Click **Run Analysis**.

Quality metrics, amplitude histograms, PSDs, and time series are all computed on the **raw, unfiltered signal**. Only the hypnospectrogram uses a 0.1&ndash;40&nbsp;Hz bandpass filter (applied only for that plot).

In [ ]:
fc_folder = FileChooser()
fc_folder.title = '<b>Select your data folder:</b>'
fc_folder.show_only_dirs = True

hypno_suffix = widgets.Text(
    value='_Hypnogram_remapped.txt',
    description='Hypno suffix:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
hypno_suffix_info = widgets.HTML(value='')

custom_stages_box = widgets.Text(
    value='',
    description='Custom stages:',
    placeholder='non-AASM labels to keep, e.g. N4 (auto-filled from config)',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
custom_stages_info = widgets.HTML(value='')


existing_reports_info = widgets.HTML(value='')
acq_info = widgets.HTML(value='')  # sampling frequency + acquisition high-pass, from EDF headers

fc_config = FileChooser(filter_pattern='*.json')
fc_config.title = '<b>Select remap/reref config JSON</b> (from select&amp;remap_channels_curry):'


def read_edf_sf_highpass(path, keep_labels=None):
    """Read sampling frequency + acquisition high-pass straight from the EDF header bytes (same
    approach as 1_inspect_edf, no MNE). Restricted to the channels in `keep_labels` (the selected
    montage) when given, else all channels. Returns (sf_str, hp_str) from the most common value
    among the kept channels: sampling frequency in Hz, and high-pass from the 'prefiltering' field
    ('none/DC' when the HP tag is absent). Raises on an unreadable header."""
    with open(path, 'rb') as f:
        f.seek(244)                                   # offset of 'duration of a data record' (8 bytes)
        dur = float(f.read(8).decode('latin-1').strip())
        n = int(f.read(4).decode('latin-1').strip())  # number of signals (channels)
        labels = [f.read(16).decode('latin-1').strip() for _ in range(n)]  # channel labels, at offset 256
        f.seek(256 + n * 136)                         # skip label/transducer/dim/phys/dig -> prefiltering
        prefilt = [f.read(80).decode('latin-1').strip() for _ in range(n)]
        spr = [f.read(8).decode('latin-1').strip() for _ in range(n)]
    idx = list(range(n)) if keep_labels is None else [i for i in range(n) if labels[i] in keep_labels]
    # Sampling frequency = samples per data record / record duration (see SPEC / 1_inspect_edf).
    sfs = []
    for i in idx:
        try:
            v = float(spr[i]) / dur
            sfs.append(f'{int(v)}' if v == int(v) else f'{v:g}')
        except (ValueError, ZeroDivisionError):
            pass
    # High-pass from the 'prefiltering' field, e.g. 'HP:0.3Hz LP:70Hz' -> 0.3; absent -> none/DC.
    hps = []
    for i in idx:
        m = re.search(r'HP[:\s]*([\d.]+)', prefilt[i], re.IGNORECASE)
        hps.append(f'{float(m.group(1)):g} Hz' if m else 'none/DC')
    # Show ALL distinct values among the kept channels (joined by ' / '), NOT the most common one:
    # a montage whose EEG channels ended up at different rates (bad export) must be visible, not
    # hidden. The caller flags any ' / ' (i.e. mixed) entry with a warning.
    sf_str = (' / '.join(sorted(set(sfs), key=float)) + ' Hz') if sfs else '?'
    hp_str = ' / '.join(sorted(set(hps))) if hps else '?'
    return sf_str, hp_str


def update_existing_reports_info(chooser):
    if not chooser.selected:
        existing_reports_info.value = ''
        hypno_suffix_info.value = ''
        acq_info.value = ''
        fc_config.reset()
        return
    data_folder = Path(chooser.selected)
    _cs = load_custom_stages(data_folder)
    custom_stages_box.value = ', '.join(_cs)
    custom_stages_info.value = (
        f'<small style="color:#2e7d32;">Custom stage(s) loaded: {", ".join(_cs)}</small>' if _cs
        else '<small style="color:#888;">No custom stages registered for this dataset.</small>'
    )
    fc_config.reset(path=str(data_folder))
    cdt_files = [f for f in sorted(data_folder.rglob('*')) if f.suffix == '.cdt' and not f.name.startswith('._')]
    n_total = len(cdt_files)
    if n_total == 0:
        existing_reports_info.value = '<small style="color:#888;">No .cdt files found in selected folder (recursive scan).</small>'
        hypno_suffix_info.value = ''
        acq_info.value = ''
        return
    reports_root = data_folder / 'reports_quality_overview'
    # "Already done" = BOTH the HTML report AND the per-file metrics TSV present. A file with
    # only one of the two (run interrupted between them, or a manual delete) is counted as a
    # mismatch and will be reprocessed when Run is pressed.
    n_existing = 0
    n_mismatch = 0
    for f in cdt_files:
        f_dir = reports_root / f.parent.relative_to(data_folder)
        has_report = (f_dir / f'{f.stem}_quality_overview.html').exists()
        has_data   = (f_dir / f'{f.stem}_quality_metrics.tsv').exists()
        if has_report and has_data:
            n_existing += 1
        elif has_report != has_data:
            n_mismatch += 1
    _mismatch_txt = (f' <span style="color:#e67e22;">({n_mismatch} inconsistent, will be reprocessed)</span>'
                     if n_mismatch else '')
    existing_reports_info.value = (
        f'<small style="color:#555;">'
        f'{n_existing} / {n_total} participant(s) already fully processed (report + data).{_mismatch_txt}</small>'
    )
    # --- Hypnogram suffix detection ---
    # For each EDF, count all .txt files whose name starts with the EDF stem (no break:
    # a participant can have both a raw and a remapped hypno).
    # Selection: among suffixes appearing for >=50% of the maximum count, prefer the
    # longest (more specific = remapped/processed version). The 50% threshold prevents
    # rare/accidental long suffixes from winning over the true candidate.
    # Tiebreaker when two candidates are equally long: highest count.
    all_txt = [f for f in data_folder.rglob('*') if f.suffix.lower() == '.txt']
    suffix_counts = {}
    for cdt in cdt_files:
        for txt in all_txt:
            if os.path.normcase(txt.name).startswith(os.path.normcase(cdt.stem)):
                suffix = txt.name[len(cdt.stem):]
                suffix_counts[suffix] = suffix_counts.get(suffix, 0) + 1
    if not suffix_counts:
        hypno_suffix_info.value = (
            '<small style="color:#e67e00;">No hypnogram files detected — '
            'check that hypnograms are present or adjust the suffix manually.</small>'
        )
    else:
        # Auto-selection skips event exports: Curry writes its scored events to
        # *_ScoredEvents_Export.txt, a .txt sitting next to the hypnograms whose suffix is
        # LONGER than _Hypnogram_remapped.txt, so 'prefer the longest' would select it and try
        # to read event lines as sleep stages. Excluding 'event' rather than requiring 'hypno'
        # leaves hypnogram naming unconstrained. Filtered out of the SELECTION only — every
        # .txt suffix stays listed below, so a wrong detection stays visible and correctable.
        sel_counts = {s: c for s, c in suffix_counts.items() if 'event' not in s.lower()}
        if not sel_counts:
            sel_counts = suffix_counts  # only event exports found: fall back rather than fail
        max_count = max(sel_counts.values())
        candidates = {s: c for s, c in sel_counts.items() if c >= max_count * 0.5}
        best_suffix, best_count = max(candidates.items(), key=lambda x: (len(x[0]), x[1]))
        hypno_suffix.value = best_suffix
        parts = [f'<b>{s}</b>&nbsp;(×{c}){"&nbsp;← selected" if s == best_suffix else ""}'
                 for s, c in sorted(suffix_counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if best_count == n_total else '#e67e00'
        hypno_suffix_info.value = (
            f'<small style="color:{color};">Detected:&nbsp;{"&nbsp;·&nbsp;".join(parts)}'
            f'&nbsp;— {best_count}/{n_total} .cdt files matched</small>'
        )

    # The acquisition line (sampling frequency / high-pass of the SELECTED channels) needs the
    # config JSON; it is filled by update_acq_info() once the config is chosen (fc_config was
    # just reset above), so here we only show a hint.
    acq_info.value = ('<small style="color:#888;">Select the config JSON above to list the '
                      'sampling frequency / high-pass of the selected channels.</small>')

fc_folder.register_callback(update_existing_reports_info)


def update_acq_info(*_):
    # Sampling frequency + acquisition high-pass of the SELECTED channels (the config 'remap' keys),
    # read straight from the EDF headers. Needs both the folder and the config JSON; grouped by
    # unique value with file counts. Registered on the config chooser below.
    if not (fc_folder.selected and fc_config.selected):
        acq_info.value = ('<small style="color:#888;">Select the data folder and the config JSON '
                          'to list the sampling frequency / high-pass of the selected channels.</small>')
        return
    try:
        data_folder = Path(fc_folder.selected)
        with open(fc_config.selected, 'r', encoding='utf-8') as _f:
            _cfg = json.load(_f)
        _cdt_files = [f for f in sorted(data_folder.rglob('*')) if f.suffix.lower() == '.edf' and not f.name.startswith('._')]
        acq_combos = {}
        for _edf in _cdt_files:
            _keep = set(_cfg.get(_edf.stem, {}).get('remap', {}).keys())
            try:
                _r = mne.io.read_raw_curry(str(_edf), preload=False, verbose='ERROR')
                _sf, _hp = f"{_r.info['sfreq']:.0f} Hz", 'none/DC'
                _key = (_sf, _hp)
            except Exception:
                _key = ('unreadable', '')
            acq_combos[_key] = acq_combos.get(_key, 0) + 1
        # A montage with several distinct rates / high-passes (sf or hp contains ' / ') is shown
        # in red — usually a bad export (channels exported at different sampling frequencies).
        acq_parts, _any_mixed = [], False
        for (_sf, _hp), _c in sorted(acq_combos.items(), key=lambda kv: -kv[1]):
            if _sf == 'unreadable':
                acq_parts.append(f'{_c} unreadable')
                continue
            _mixed = (' / ' in _sf) or (' / ' in _hp)
            _any_mixed = _any_mixed or _mixed
            _txt = f'{_sf} · high-pass {_hp} (×{_c})'
            acq_parts.append(f'<b style="color:#c0392b;">⚠ {_txt}</b>' if _mixed else _txt)
        _note = (' &nbsp;&mdash;&nbsp; <b style="color:#c0392b;">⚠ a montage mixes several sampling '
                 'frequencies / high-passes (check the export!)</b>') if _any_mixed else ''
        acq_info.value = (
            '<small style="color:#555;">Selected channels (Curry header): '
            + '&nbsp;&nbsp;·&nbsp;&nbsp;'.join(acq_parts) + _note + '</small>'
        )
    except Exception as _acq_err:
        acq_info.value = f'<small style="color:#e67e00;">Could not read acquisition info: {_acq_err}</small>'

fc_config.register_callback(update_acq_info)

skip_existing = widgets.Checkbox(
    value=True,
    description='Skip participants with an existing report',
    style={'description_width': 'initial'}
)

thresh_flat = widgets.BoundedFloatText(
    value=3.5, min=0.0, max=100.0, step=0.5,
    description='flat_pct (%) >',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_peaks = widgets.BoundedIntText(
    value=2, min=2, max=10, step=1,
    description='n_peaks >=',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_std_low = widgets.BoundedFloatText(
    value=5.0, min=0.0, max=500.0, step=0.5,
    description='std_uV (µV) <',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)
thresh_hist_extreme = widgets.BoundedFloatText(
    value=1.0, min=0.0, max=100.0, step=0.1,
    description='hist_extreme_pct (%) >',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='300px')
)

# ---- Optional resampling (speeds up processing; off by default) ----
cb_resample = widgets.Checkbox(
    value=False, description='Activate resampling',
    style={'description_width': 'initial'}
)
txt_target_freq = widgets.BoundedIntText(
    value=256, min=1, max=10000, step=1,
    description='Target frequency (Hz) :',
    style={'description_width': '180px'},
    # Initial visibility follows the checkbox default (same reason as the high-pass box below).
    layout=widgets.Layout(width='320px', display='' if cb_resample.value else 'none')
)

def _toggle_resample(change):
    txt_target_freq.layout.display = '' if change['new'] else 'none'

cb_resample.observe(_toggle_resample, names='value')

# ---- Optional high-pass (analysis only; off by default) ----
# EDFs are usually already AC-coupled (their hardware high-pass is shown in the acquisition
# line above). Applying one here is mainly to HARMONISE a heterogeneous dataset to a common
# corner — choose a target >= the max acquisition high-pass shown. Applied before resampling.
hp_check = widgets.Checkbox(
    value=True, description='Activate high-pass (DC-coupled Curry data)',
    style={'description_width': 'initial'}
)
hp_freq = widgets.BoundedFloatText(
    value=0.1, min=0.01, max=5.0, step=0.05,
    description='High-pass (Hz) :',
    style={'description_width': '180px'},
    # Initial visibility follows the checkbox default (the Curry twin ticks it ON), otherwise
    # the box would stay hidden until the user toggles the checkbox off and on again.
    layout=widgets.Layout(width='320px', display='' if hp_check.value else 'none')
)

def _toggle_highpass(change):
    hp_freq.layout.display = '' if change['new'] else 'none'

hp_check.observe(_toggle_highpass, names='value')


def thresh_row(w, desc):
    return widgets.HBox([
        w,
        widgets.HTML(
            f'<small style="color:#555;margin-left:12px;line-height:32px;">{desc}</small>'
        )
    ])


display(
    HTML('<h3 style="margin-top:4px;">&#128193; Data</h3>'),
    fc_folder,
    hypno_suffix,
    hypno_suffix_info,
    custom_stages_box,
    custom_stages_info,
    fc_config,
    existing_reports_info,
    skip_existing,
    HTML('<h3 style="margin-top:20px;">&#128295; Signal preparation '
         '<small style="color:#888;font-weight:normal;">(applied for this analysis only — not saved)</small></h3>'),
    acq_info,
    widgets.HBox([cb_resample, txt_target_freq]),
    widgets.HBox([hp_check, hp_freq]),
    HTML(
        '<h3 style="margin-top:20px;">&#9881; Quality thresholds</h3>'
        '<small style="color:#555;">Channels exceeding any threshold will be flagged '
        'in the report and in <em>dataset_overview.html</em>.</small>'
    ),
    widgets.VBox([
        thresh_row(thresh_flat,
                   'Fraction of consecutive equal samples — detects flat segments and dead channels'),
        thresh_row(thresh_peaks,
                   'Peaks in amplitude distribution — ≥2: DC drift (bimodal); ≥3: coarse quantization'),
        thresh_row(thresh_std_low,
                   'Signal std dev — very low std (< 5 µV) flags a dead or disconnected channel'),
        thresh_row(thresh_hist_extreme,
                   'Fraction in outermost histogram bins — detects in-range clipping (within EDF range)'),
    ])
)

In [ ]:
btn_run = widgets.Button(
    description='▶  Run',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
progress = widgets.IntProgress(
    min=0, max=1, value=0, bar_style='info',
    layout=widgets.Layout(width='500px')
)
progress_label = widgets.Label(value='')

# --- Pipeline progress (second bar): spans ONE participant's whole run, so a long file is
# not mistaken for a crash. Its scale is in arbitrary 'time-cost' units (below) so the bar
# fill AND the 3-segment legend under it are sized ~proportionally to each phase's duration.
# Loading is a one-time cost; the per-channel analysis and (heavier) report-rendering costs
# scale with the channel count. Tune these if the segment widths feel off for your data.
COST_LOAD_DATA    = 8    # EDF read — one-time per file
COST_HIGHPASS     = 3    # high-pass filter — one-time, only when enabled
COST_RESAMPLE     = 4    # resampling — one-time, only when enabled
COST_ANALYSE_CH   = 1    # per channel — metric / histogram computation
COST_RENDER_CH    = 3    # per channel — report figures (the slowest step)


def build_phase_legend(cost_load, cost_analyse, cost_report):
    """3-segment strip drawn under the pipeline bar; segment widths ≈ each phase's share of
    the run time (Load / Per-channel / Report). Rebuilt per participant (widths depend on
    the channel count), and aligned to the 500px bar so the fill matches the segments."""
    total = max(1, cost_load + cost_analyse + cost_report)
    def seg(cost, label, bg, tip, last=False):
        border = '' if last else 'border-right:1px solid #bbb;'
        return (f'<div title="{tip}" style="width:{100 * cost / total:.1f}%;{border}'
                f'text-align:center;background:{bg};overflow:hidden;white-space:nowrap;">{label}</div>')
    return (
        '<div style="display:flex;width:500px;height:14px;font-size:9px;line-height:14px;'
        'color:#444;border:1px solid #bbb;border-top:none;border-radius:0 0 3px 3px;overflow:hidden;">'
        + seg(cost_load, 'Load', '#e7ecff', 'Loading + resampling')
        + seg(cost_analyse, 'Per-channel', '#e7f7e7', 'Per-channel analysis')
        + seg(cost_report, 'Report', '#ffeaea', 'Report building', last=True)
        + '</div>'
    )


progress_ch = widgets.IntProgress(
    min=0, max=1, value=0, bar_style='',
    layout=widgets.Layout(width='500px', margin='0')
)
progress_ch_label = widgets.Label(value='')
# margin='0' + the legend glued directly under the bar (see the display VBox) so the 3 phase
# segments read as the bar's own labelled track.
phase_legend = widgets.HTML(value='', layout=widgets.Layout(margin='0'))
out = widgets.Output()


def run_analysis(btn):
    btn.disabled = True
    out.clear_output()
    try:

        # --- Validate inputs ---
        if not fc_folder.selected:
            with out:
                print('ERROR: Please select a data folder.')
            return
        if not fc_config.selected:
            with out:
                print('ERROR: Please select the remap/reref config JSON.')
            return

        data_folder = Path(fc_folder.selected)
        config_path = Path(fc_config.selected)
        reports_root = data_folder / 'reports_quality_overview'
        reports_root.mkdir(exist_ok=True)
        custom_stages = parse_custom_field(custom_stages_box.value)

        try:
            with open(config_path, 'r', encoding='utf-8') as f:
                config_dict = json.load(f)
        except Exception as e:
            with out:
                print(f'ERROR loading config: {e}')
            return

        cdt_files = [f for f in sorted(data_folder.rglob('*')) if f.suffix == '.cdt' and not f.name.startswith('._')]
        if not cdt_files:
            with out:
                print(f'No .cdt files found in {data_folder}')
            return

        progress.max = len(cdt_files)
        progress.value = 0
        hypno_lookup = {os.path.normcase(p.name): p for p in data_folder.rglob('*') if p.suffix.lower() == '.txt'}

        thresholds = {
            'flat_pct': thresh_flat.value,
            'n_peaks': thresh_peaks.value,
            'std_low': thresh_std_low.value,
            'hist_extreme_pct': thresh_hist_extreme.value,
        }
        do_resample = cb_resample.value
        target_freq = int(txt_target_freq.value) if do_resample else None
        do_highpass = hp_check.value
        hp_freq_val = float(hp_freq.value) if do_highpass else None
        # Build per-report interpretations: threshold-based metrics show the actual
        # threshold value used for this run (set manually in the notebook widgets).
        interpretations = dict(METRIC_INTERPRETATIONS)
        interpretations.update({
            'Flat signal (%)': f'flag if&nbsp;&gt;&nbsp;{thresholds["flat_pct"]:g}%&nbsp;&mdash; threshold defined manually in the notebook ; typical threshold is ??',
            'Extreme histogram (%)': f'flag if&nbsp;&gt;&nbsp;{thresholds["hist_extreme_pct"]:g}%&nbsp;&mdash; threshold defined manually in the notebook ; typical threshold is ??',
        })

        rows_summary = []
        rows_stage_summary = []
        failed = []
        attempted_ids = set()  # file_ids actually processed (not skipped)
        t_start = time.time()

        for i, cdt_path in enumerate(cdt_files):
            file_id = cdt_path.stem
            progress.value = i
            progress_label.value = f'Participant {file_id}  ({i + 1}/{len(cdt_files)})'
            progress_ch.value = 0
            progress_ch_label.value = ''
            phase_legend.value = ''

            # Mirror the EDF subfolder structure under reports_root.
            cdt_relative = cdt_path.parent.relative_to(data_folder)
            cdt_reports_dir = reports_root / cdt_relative
            report_path = cdt_reports_dir / f'{file_id}_quality_overview.html'
            metrics_path = cdt_reports_dir / f'{file_id}_quality_metrics.tsv'

            # Skip only when BOTH the report and the per-file metrics TSV exist, so a run
            # interrupted between the two never leaves a file permanently skipped with its
            # rows missing from quality_summary.tsv. A mismatch is warned about + reprocessed.
            _has_report = report_path.exists()
            _has_data   = metrics_path.exists()
            if skip_existing.value and _has_report and _has_data:
                with out:
                    print(f'⤼ {file_id} ({i + 1}/{len(cdt_files)}) — skipped (report + data exist)')
                continue
            if skip_existing.value and (_has_report != _has_data):
                with out:
                    print(f'⚠ {file_id} — '
                          + ('report exists but metrics table missing' if _has_report else 'metrics table exists but report missing')
                          + ' — reprocessing to restore consistency.')

            attempted_ids.add(file_id)
            cdt_reports_dir.mkdir(parents=True, exist_ok=True)
            file_rows = []        # this file's per-channel metric rows → {file_id}_quality_metrics.tsv
            file_stage_rows = []  # this file's per-stage rows → {file_id}_quality_by_stage.tsv (if hypnogram)

            # --- Load hypnogram ---
            progress_ch_label.value = f'{file_id} · loading hypnogram…'
            _hypno_name = f'{file_id}{hypno_suffix.value}'
            hypno_path = hypno_lookup.get(os.path.normcase(_hypno_name), cdt_path.parent / _hypno_name)
            hypno_vec = None
            hypno_warning = None
            hypno_warning_plain = None
            if not hypno_path.exists():
                hypno_warning = (
                    f'Hypnogram not found: <b>{hypno_path.name}</b>. '
                    f'Spectrogram skipped. Possible causes: '
                    f'(1) the <i>3_remap_hypno_voila</i> notebook was not run or did not complete; '
                    f'(2) the hypnogram suffix above does not match the actual file suffix.'
                )
                hypno_warning_plain = f'Hypnogram not found: {hypno_path.name} (spectrogram skipped): check that the suffix is correct or that 3_remap_hypno_voila was run.'
            else:
                try:
                    expert_hypno = np.loadtxt(str(hypno_path), dtype=str).astype('<U10')
                    stage_map = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
                    for _i, _cs in enumerate(custom_stages):
                        stage_map[_cs] = 5 + _i
                    hypno_vec = np.full(len(expert_hypno), np.nan)
                    for stage, val in stage_map.items():
                        hypno_vec[expert_hypno == stage] = val
                    # MT (movement time) is tolerated: YASA treats it as artifact (NaN), no warning needed.
                    # Any other unrecognised label means the hypnogram was not converted to AASM convention.
                    _tolerated = {'MT'}
                    _unknown_mask = np.array([s not in stage_map and s not in _tolerated for s in expert_hypno])
                    _unknown_labels = sorted(set(expert_hypno[_unknown_mask]))
                    _n_unknown = int(_unknown_mask.sum())
                    _pct_unknown = _n_unknown / len(expert_hypno) * 100
                    if _pct_unknown > 10:
                        hypno_vec = None
                        hypno_warning = (
                            f'Hypnogram labels not AASM: {_n_unknown}/{len(expert_hypno)} epochs '
                            f'({_pct_unknown:.1f}%) have unrecognised labels '
                            f'({", ".join(_unknown_labels)}). Spectrogram skipped. '
                            f'Run <i>3_remap_hypno_voila</i> to convert labels to W/N1/N2/N3/R.'
                        )
                        hypno_warning_plain = (
                            f'Hypnogram labels not AASM: {_n_unknown}/{len(expert_hypno)} epochs '
                            f'({_pct_unknown:.1f}%) have unrecognised labels '
                            f'({", ".join(_unknown_labels)}). Spectrogram skipped.'
                        )
                    elif _unknown_labels:
                        hypno_warning = (
                            f'{_n_unknown} epoch(s) ({_pct_unknown:.1f}%) have unrecognised label(s): '
                            f'{", ".join(_unknown_labels)}. These epochs will be treated as artifact '
                            f'in the spectrogram.'
                        )
                        hypno_warning_plain = (
                            f'{_n_unknown} epoch(s) ({_pct_unknown:.1f}%) have unrecognised label(s): '
                            f'{", ".join(_unknown_labels)}. Treated as artifact in the spectrogram.'
                        )
                except Exception as e:
                    hypno_warning = f'Error loading hypnogram: {e}'
                    hypno_warning_plain = f'Error loading hypnogram: {e}'

            # --- Check config ---
            if file_id not in config_dict:
                with out:
                    print(f'WARNING: {file_id} not in config JSON, skipping.')
                failed.append({'file_id': file_id, 'reason': 'not in config JSON'})
                continue

            sub_config = config_dict[file_id]
            selected_channels = list(sub_config['remap'].keys())

            # Set up the pipeline bar for this participant. Channel count is provisional here
            # (from the config); it is corrected to the real count once the EDF is loaded,
            # just before the per-channel analysis. Costs come from the constants above.
            _n_ch_est = len(selected_channels)
            _cost_load = COST_LOAD_DATA + (COST_HIGHPASS if do_highpass else 0) + (COST_RESAMPLE if do_resample else 0)
            progress_ch.max = _cost_load + (COST_ANALYSE_CH + COST_RENDER_CH) * _n_ch_est
            progress_ch.value = 0
            phase_legend.value = build_phase_legend(_cost_load, COST_ANALYSE_CH * _n_ch_est, COST_RENDER_CH * _n_ch_est)

            progress_ch_label.value = f'{file_id} · loading EDF signal…'
            # --- Load Curry signal (pick selected channels before loading to limit memory use) ---
            try:
                raw = mne.io.read_raw_curry(str(cdt_path), preload=False, verbose='ERROR')
                _present = [ch for ch in selected_channels if ch in raw.ch_names]
                raw.pick(_present)
                raw.load_data()
            except Exception as e:
                with out:
                    print(f'ERROR loading {file_id}: {e}')
                failed.append({'file_id': file_id, 'reason': f'Curry loading: {e}'})
                continue

            raw.rename_channels({k: v for k, v in sub_config['remap'].items() if k in raw.ch_names})
            sf = raw.info['sfreq']
            progress_ch.value = COST_LOAD_DATA   # segment 1: data loaded

            # --- Optional high-pass (analysis only; applied BEFORE resampling) ---
            # Removes DC offset / slow drift on the full-rate signal. Done before resampling so
            # the resampler's anti-alias filter does not ring on a large DC/drift. Off by default;
            # useful to harmonise a heterogeneous dataset (or for DC-coupled data, e.g. Curry).
            if do_highpass and hp_freq_val is not None:
                progress_ch_label.value = f'{file_id} · high-pass {hp_freq_val:g} Hz…'
                try:
                    raw.filter(l_freq=hp_freq_val, h_freq=None, verbose=False)
                    progress_ch.value = COST_LOAD_DATA + COST_HIGHPASS   # segment 1: high-pass done
                except Exception as _hp_err:
                    with out:
                        print(f'  ⚠ {file_id}: high-pass ({hp_freq_val:g} Hz) failed: {_hp_err}')

            # --- Optional resampling (speeds up processing; off by default) ---
            # Resampling changes the signal, so shape metrics (flat_pct, histograms) and the
            # spectrogram are computed on the resampled data. Guarded to never upsample.
            if do_resample and target_freq is not None and target_freq < sf:
                progress_ch_label.value = f'{file_id} · resampling to {target_freq} Hz…'
                try:
                    raw.resample(target_freq, npad='auto', verbose=False)
                    sf = raw.info['sfreq']
                    progress_ch.value = _cost_load   # segment 1: resampling done
                except Exception as _rs_err:
                    with out:
                        print(f'  ⚠ {file_id}: resample to {target_freq} Hz failed: {_rs_err}')

            # --- Hypnogram length check ---
            if hypno_vec is not None:
                expected = int(np.floor(raw.n_times / sf / 30))
                n_hyp = len(hypno_vec)
                overhang = n_hyp - expected
                if 0 < overhang <= 1:
                    # Recording ends mid-epoch: the last epoch is partial (< 30 s) but was still
                    # scored. Drop the overhanging score (no full 30 s of signal to analyse).
                    last_sec = raw.n_times / sf - expected * 30
                    expert_hypno = expert_hypno[:expected]
                    hypno_vec = hypno_vec[:expected]
                    hypno_warning = (
                        f'Recording ends mid-epoch: the last epoch is only {last_sec:.1f}s '
                        f'(< 30s) so its score is skipped. Hypnogram trimmed from {n_hyp} to '
                        f'{expected} epochs to match the complete 30s signal epochs.'
                    )
                    hypno_warning_plain = (
                        f'Last epoch {last_sec:.1f}s (< 30s) skipped; hypnogram trimmed '
                        f'{n_hyp} -> {expected} to match complete 30s signal epochs.'
                    )
                elif overhang != 0:
                    hypno_warning = (
                        f'Length mismatch: EEG has {expected} 30s-epochs, '
                        f'hypnogram has {n_hyp}. Spectrogram skipped.'
                    )
                    hypno_warning_plain = (
                        f'Length mismatch: EEG has {expected} 30s-epochs, '
                        f'hypnogram has {n_hyp}. Spectrogram skipped.'
                    )
                    hypno_vec = None

            # --- Compute metrics for all channels (unfiltered signal) ---
            # Memory: a full-night, high-rate recording with many channels (e.g. 32 EEG @ 1024 Hz)
            # is ~10 GiB as one float64 array. We never materialise the whole thing — each channel
            # is read on demand with raw.get_data(picks=[ci]) (a cheap slice of the preloaded data),
            # and the shared amplitude limits are derived from the per-channel p99.9 metric below.

            ch_metrics = {}
            ch_flags = {}
            epoch_len = int(30 * sf)
            # Finalize the pipeline bar now the real channel count is known (rebuild the legend
            # if it differs from the config estimate). Analysis fills segment 2, one tick/channel.
            n_ch = len(raw.ch_names)
            _cost_analyse = COST_ANALYSE_CH * n_ch
            _cost_render = COST_RENDER_CH * n_ch
            progress_ch.max = _cost_load + _cost_analyse + _cost_render
            phase_legend.value = build_phase_legend(_cost_load, _cost_analyse, _cost_render)
            progress_ch.value = _cost_load
            for ci, ch in enumerate(raw.ch_names):
                progress_ch.value = _cost_load + COST_ANALYSE_CH * (ci + 1)
                progress_ch_label.value = f'{file_id} · analysing {ch} ({ci + 1}/{n_ch})'
                sig_uV = raw.get_data(picks=[ci])[0] * 1e6
                m = compute_signal_metrics(sig_uV)
                ch_metrics[ch] = m
                ch_flags[ch] = flag_channel(m, thresholds)
                file_rows.append({
                    'file_id': file_id,
                    'channel': ch,
                    'mean_uV': m['mean_uV'],
                    'std_uV': m['std_uV'],
                    'kurtosis': m['kurtosis'],
                    'skewness': m['skewness'],
                    'p99_abs_uV': m['p99_abs_uV'],
                    'p999_abs_uV': m['p999_abs_uV'],
                    'flat_pct': m['flat_pct'],
                    'hist_extreme_pct': m['hist_extreme_pct'],
                    'n_peaks': m['n_peaks'],
                    'suspect_reason': '; '.join(ch_flags[ch]),
                    'exclude': bool(ch_flags[ch]),
                })

                # Per-stage key metrics: split the signal into 30s epochs by hypnogram stage.
                # Skipped if no valid hypnogram (not found, length mismatch, or bad labels).
                if hypno_vec is not None:
                    _stage_pairs = [(0, 'W'), (1, 'N1'), (2, 'N2'), (3, 'N3'), (4, 'R')]
                    _stage_pairs += [(5 + _i, _cs) for _i, _cs in enumerate(custom_stages)]
                    for stage_int, stage_str in _stage_pairs:
                        stage_indices = np.where(hypno_vec == stage_int)[0]
                        if len(stage_indices) == 0:
                            continue
                        stage_samples = []
                        for ep_idx in stage_indices:
                            start_samp = ep_idx * epoch_len
                            end_samp = start_samp + epoch_len
                            if end_samp <= len(sig_uV):
                                stage_samples.append(sig_uV[start_samp:end_samp])
                        if not stage_samples:
                            continue
                        stage_sig = np.concatenate(stage_samples)
                        sm = compute_signal_metrics(stage_sig)
                        file_stage_rows.append({
                            'file_id': file_id,
                            'channel': ch,
                            'stage': stage_str,
                            'mean_uV': sm['mean_uV'],
                            'std_uV': sm['std_uV'],
                            'flat_pct': sm['flat_pct'],
                            'hist_extreme_pct': sm['hist_extreme_pct'],
                            'p99_abs_uV': sm['p99_abs_uV'],
                            'p999_abs_uV': sm['p999_abs_uV'],
                        })

            # Shared time-series / histogram amplitude limits from the per-channel p99.9 metric
            # (avoids materialising the whole recording just to take a global percentile).
            # DC-coupled data carries no export clipping, so drift/artifacts can push the
            # p99.9 autoscale far past physiological range and crush the real EEG. Cap the
            # shared time-series / histogram amplitude limit at a wide physiological ceiling
            # (uV); clean low-amplitude channels still auto-zoom below the cap.
            DISPLAY_YLIM_UV = 500.0
            _p999 = [m['p999_abs_uV'] for m in ch_metrics.values()]
            y_lim_ts = min(float(max(_p999)), DISPLAY_YLIM_UV) if _p999 else 1.0
            x_lim_hist = y_lim_ts

            # Shared Y-axis for histograms: computed from non-suspect channels only.
            # A flat or dead channel concentrates all samples in 1-2 bins, giving a
            # y_max that would crush the distributions of healthy channels.
            # Flagged channels are plotted with their own auto-scale (y_max=None) to
            # avoid their distribution being clipped by the healthy-channel scale.
            healthy_chs = [ch for ch, f in ch_flags.items() if not f]
            if healthy_chs:
                hist_y_max = max(float(np.max(ch_metrics[ch]['histo'])) for ch in healthy_chs)
            elif ch_metrics:
                hist_y_max = max(float(np.max(m['histo'])) for m in ch_metrics.values())
            else:
                hist_y_max = None

            progress_ch_label.value = f'{file_id} · building overview…'
            # --- Build mne.Report ---
            report = mne.Report(title=f'{file_id} — Quality Overview', verbose=False)

            # Overview section: channel flag summary
            flagged = {ch: f for ch, f in ch_flags.items() if f}
            if flagged:
                items = ''.join(
                    f'<li><b>{ch}</b>: {", ".join(f)}</li>' for ch, f in flagged.items()
                )
                flag_html = (
                    '<div style="background:#f8d7da;padding:10px;'
                    'border-left:4px solid #dc3545;border-radius:4px;margin-bottom:12px;">'
                    f'<b>⚠ Suspicious channels ({len(flagged)}/{len(raw.ch_names)}):'
                    f'</b><ul>{items}</ul></div>'
                )
            else:
                flag_html = (
                    '<div style="background:#d4edda;padding:10px;'
                    'border-left:4px solid #28a745;border-radius:4px;margin-bottom:12px;">'
                    '<b>✓ All channels within thresholds</b></div>'
                )
            if hypno_warning:
                flag_html += (
                    '<div style="background:#fff3cd;padding:8px;'
                    'border-left:4px solid #ffc107;border-radius:4px;margin-bottom:12px;">'
                    f'⚠ {hypno_warning}</div>'
                )
            report.add_html(html=flag_html, title='Overview', section='Overview')

            # Overview overlays: all channels superimposed (one colour each) + across-channel table.
            try:
                fig_ov_ts = plot_overlay_timeseries(raw, sf, y_lim=y_lim_ts)
                report.add_figure(fig=fig_ov_ts, title='All channels \u2014 time series (butterfly)',
                                  section='Overview', image_format='PNG')
                plt.close(fig_ov_ts)
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview time-series failed: {_ov_err}')
            try:
                fig_ov_psd = plot_overlay_psd(raw, sf)
                report.add_figure(fig=fig_ov_psd, title='All channels \u2014 PSD',
                                  section='Overview', image_format='PNG')
                plt.close(fig_ov_psd)
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview PSD failed: {_ov_err}')
            try:
                fig_ov_dist = plot_overlay_distribution(raw, x_lim=x_lim_hist)
                report.add_figure(fig=fig_ov_dist, title='All channels \u2014 amplitude distribution',
                                  section='Overview', image_format='PNG')
                plt.close(fig_ov_dist)
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview distribution failed: {_ov_err}')
            try:
                report.add_html(html=build_avg_metrics_html(ch_metrics, raw.ch_names),
                                title='Metrics averaged across channels', section='Overview')
            except Exception as _ov_err:
                with out:
                    print(f'  \u26a0 {file_id}: overview metrics table failed: {_ov_err}')

            # One section per channel: histogram + PSD + time series + spectrogram + metrics table.
            # Report = segment 3 of the pipeline bar: it advances once per rendered channel (the
            # bar keeps moving forward continuously — no per-channel reset like the old two-bar UI).
            _report_base = _cost_load + _cost_analyse
            for ci, ch in enumerate(raw.ch_names):
                progress_ch_label.value = f'{file_id} · building report {ch} ({ci + 1}/{n_ch})'
                progress_ch.value = _report_base + COST_RENDER_CH * (ci + 1)
                m = ch_metrics[ch]
                flags = ch_flags[ch]
                sig_uV = raw.get_data(picks=[ci])[0] * 1e6

                # Flagged channels use their own auto-scale; healthy channels share hist_y_max.
                y_max_ch = hist_y_max if not flags else None
                try:
                    fig = plot_histogram_figure(m, ch, y_max=y_max_ch, x_lim=x_lim_hist)
                    report.add_figure(fig=fig, title=f'{ch} — histogram', section=ch,
                                      image_format='PNG')
                    plt.close(fig)
                except Exception as _hist_err:
                    report.add_html(html=f'<p>Histogram generation failed: {_hist_err}</p>',
                                    title=f'{ch} — histogram', section=ch)
                    with out:
                        print(f'  ⚠ {file_id}/{ch}: histogram error: {_hist_err}')

                try:
                    fig_psd = plot_psd_figure(sig_uV, sf)
                    report.add_figure(fig=fig_psd, title=f'{ch} — PSD', section=ch,
                                      image_format='PNG')
                    plt.close(fig_psd)
                except Exception as _psd_err:
                    report.add_html(
                        html=f'<p>PSD generation failed: {_psd_err}</p>',
                        title=f'{ch} — PSD', section=ch
                    )
                    with out:
                        print(f'  ⚠ {file_id}/{ch}: PSD error: {_psd_err}')

                try:
                    fig_ts = plot_timeseries_figure(sig_uV, sf, y_lim_ts)
                    report.add_figure(fig=fig_ts, title=f'{ch} — time series', section=ch,
                                      image_format='PNG')
                    plt.close(fig_ts)
                except Exception as _ts_err:
                    report.add_html(html=f'<p>Time series generation failed: {_ts_err}</p>',
                                    title=f'{ch} — time series', section=ch)
                    with out:
                        print(f'  ⚠ {file_id}/{ch}: time series error: {_ts_err}')

                # Spectrogram: bandpass-filtered copy (0.1–40 Hz) computed just before plotting.
                if hypno_vec is not None:
                    try:
                        sig_filt = mne.filter.filter_data(
                            sig_uV[np.newaxis, :], sfreq=sf, l_freq=0.1, h_freq=40,
                            method='fir', phase='zero-double', fir_window='hamming', verbose=False
                        )[0]
                        fig_spec = plot_hypnospectrogram(sig_filt, sf, expert_hypno, custom_stages, fmin=0.5, fmax=40)
                        fig_spec.set_size_inches(10, 3.5)
                        for ax in fig_spec.axes:
                            ax.tick_params(labelsize=8)
                            ax.xaxis.label.set_fontsize(11)
                            ax.yaxis.label.set_fontsize(11)
                        fig_spec.tight_layout()
                        report.add_figure(fig=fig_spec, title=f'{ch} — spectrogram', section=ch,
                                          image_format='PNG')
                        plt.close(fig_spec)
                    except Exception as _spec_err:
                        report.add_html(
                            html=f'<p>Spectrogram generation failed: {_spec_err}</p>',
                            title=f'{ch} — spectrogram', section=ch
                        )
                        with out:
                            print(f'  ⚠ {file_id}/{ch}: spectrogram error: {_spec_err}')
                else:
                    report.add_html(
                        html='<p>No spectrogram: hypnogram missing or incompatible with EEG length.</p>',
                        title=f'{ch} — spectrogram', section=ch
                    )

                flag_bg = '#f8d7da' if flags else '#f0f0f0'
                flag_text = '; '.join(flags) if flags else 'no flags'
                rows_html = ''.join([
                    f'<tr><td style="padding:4px 10px;">{label}</td>'
                    f'<td style="padding:4px 10px;text-align:right;">{val}</td>'
                    f'<td style="padding:4px 10px;color:#555;font-size:0.85em;">'
                    f'{interpretations.get(label, "")}</td></tr>'
                    for label, val in [
                        ('Mean', f"{m['mean_uV']:.2f} µV"),
                        ('Std dev', f"{m['std_uV']:.2f} µV"),
                        ('p99 |amplitude|', f"{m['p99_abs_uV']:.1f} µV"),
                        ('p99.9 |amplitude|', f"{m['p999_abs_uV']:.1f} µV"),
                        ('Kurtosis (Fischer)', f"{m['kurtosis']:.2f}"),
                        ('Skewness', f"{m['skewness']:.3f}"),
                        ('Flat signal (%)', f"{m['flat_pct']:.2f}%"),
                        ('Extreme histogram (%)', f"{m['hist_extreme_pct']:.3f}%"),
                        ('n_peaks (distribution)', str(m['n_peaks'])),
                    ]
                ])
                metrics_html = (
                    f'<div style="background:{flag_bg};padding:8px;border-radius:4px;'
                    f'margin-bottom:8px;"><b>Flags:</b> {flag_text}</div>'
                    '<table style="border-collapse:collapse;font-size:0.9em;width:100%;">'
                    '<tr style="background:#eee;">'
                    '<th style="padding:4px 10px;text-align:left;">Metric</th>'
                    '<th style="padding:4px 10px;text-align:right;">Value</th>'
                    '<th style="padding:4px 10px;text-align:left;">Interpretation</th></tr>'
                    f'{rows_html}</table>'
                )
                report.add_html(html=metrics_html, title=f'{ch} — metrics', section=ch)

            # --- Write per-file TSVs BEFORE the HTML report ---
            # These per-file tables are the durable cumulative data: quality_summary.tsv is
            # rebuilt from all of them by glob after the loop. Writing them before report.save
            # guarantees 'report exists => data exists', so the skip gate is interruption-safe.
            try:
                if file_rows:
                    pd.DataFrame(file_rows).to_csv(str(metrics_path), sep='\t', index=False)
                if file_stage_rows:
                    pd.DataFrame(file_stage_rows).to_csv(
                        str(cdt_reports_dir / f'{file_id}_quality_by_stage.tsv'), sep='\t', index=False)
            except Exception as _e:
                with out:
                    print(f'  ⚠ {file_id}: could not write per-file metrics TSV: {_e}')
            # Keep the in-memory accumulation for the end-of-run summary counts below.
            rows_summary.extend(file_rows)
            rows_stage_summary.extend(file_stage_rows)

            progress_ch_label.value = f'{file_id} · saving report…'
            report.save(str(report_path), overwrite=True, open_browser=False, verbose=False)
            n_flagged = sum(1 for f in ch_flags.values() if f)
            progress_ch_label.value = f'{file_id} · done ✓'
            with out:
                print(f'✓ {file_id} — processed ({n_flagged}/{len(raw.ch_names)} channel(s) flagged)  →  {report_path.name}')
                if hypno_warning_plain:
                    print(f'  ⚠ {hypno_warning_plain}')

        # --- Rebuild cumulative TSVs from the per-file tables on disk ---
        # quality_summary.tsv and quality_summary_by_stage.tsv are regenerated from ALL
        # per-file *_quality_metrics.tsv / *_quality_by_stage.tsv found under reports_root
        # (any subfolder), so a file processed in an earlier run and skipped in this one is
        # never dropped from the cumulative tables. df_new / df_stage_new (this run's rows)
        # are kept below only for the end-of-run summary counts.
        df_new = pd.DataFrame(rows_summary)
        df_stage_new = pd.DataFrame(rows_stage_summary)
        summary_path = reports_root / 'quality_summary.tsv'
        stage_summary_path = reports_root / 'quality_summary_by_stage.tsv'

        metrics_frames = []
        for _p in sorted(reports_root.rglob('*_quality_metrics.tsv')):
            try:
                metrics_frames.append(pd.read_csv(str(_p), sep='\t', dtype={'file_id': str}))
            except Exception as _e:
                with out:
                    print(f'  ⚠ could not read {_p.name}: {_e}')
        if metrics_frames:
            df_summary_all = pd.concat(metrics_frames, ignore_index=True).sort_values(['file_id', 'channel']).reset_index(drop=True)
            df_summary_all.to_csv(str(summary_path), sep='\t', index=False)

        stage_frames = []
        for _p in sorted(reports_root.rglob('*_quality_by_stage.tsv')):
            try:
                stage_frames.append(pd.read_csv(str(_p), sep='\t', dtype={'file_id': str}))
            except Exception as _e:
                with out:
                    print(f'  ⚠ could not read {_p.name}: {_e}')
        if stage_frames:
            df_stage_all = pd.concat(stage_frames, ignore_index=True).sort_values(['file_id', 'channel', 'stage']).reset_index(drop=True)
            df_stage_all.to_csv(str(stage_summary_path), sep='\t', index=False)

        # Regenerate dataset_overview.html from the full cumulative quality_summary.tsv
        # so the overview reflects all participants processed to date, not just this run.
        if summary_path.exists():
            try:
                _df_stage = (
                    pd.read_csv(str(stage_summary_path), sep='\t', dtype={'file_id': str})
                    if stage_summary_path.exists() else None
                )
                generate_dataset_overview(
                    pd.read_csv(str(summary_path), sep='\t'), reports_root, df_stage=_df_stage, custom_stages=custom_stages
                )
            except Exception as _ov_err:
                with out:
                    print(f'⚠ dataset_overview.html generation failed: {_ov_err}')

        failed_path = reports_root / 'failed_files.tsv'
        if failed:
            df_failed_new = pd.DataFrame(failed)
            if failed_path.exists() and attempted_ids:
                # dtype={'file_id': str}: same reason as quality_summary.tsv above.
                df_failed_existing = pd.read_csv(str(failed_path), sep='\t', dtype={'file_id': str})
                df_failed_existing = df_failed_existing[~df_failed_existing['file_id'].astype(str).map(os.path.normcase).isin({os.path.normcase(s) for s in attempted_ids})]
                df_failed = pd.concat([df_failed_existing, df_failed_new], ignore_index=True)
            else:
                df_failed = df_failed_new
            df_failed.to_csv(str(failed_path), sep='\t', index=False)
        elif failed_path.exists() and attempted_ids:
            # A file that was previously failing and was re-attempted successfully: remove its entry.
            # dtype={'file_id': str}: same reason as quality_summary.tsv above.
            df_failed_existing = pd.read_csv(str(failed_path), sep='\t', dtype={'file_id': str})
            df_failed_cleaned = df_failed_existing[~df_failed_existing['file_id'].astype(str).map(os.path.normcase).isin({os.path.normcase(s) for s in attempted_ids})]
            if df_failed_cleaned.empty:
                failed_path.unlink()
            elif len(df_failed_cleaned) < len(df_failed_existing):
                df_failed_cleaned.to_csv(str(failed_path), sep='\t', index=False)

        progress.value = len(cdt_files)
        progress_ch.value = progress_ch.max
        progress_label.value = 'Done.'
        progress_ch_label.value = ''

        elapsed = time.time() - t_start
        mins, secs = divmod(elapsed, 60)
        elapsed_str = f'{int(mins)} min {secs:.0f} s' if mins >= 1 else f'{secs:.1f} s'

        with out:
            if not df_new.empty:
                n_files_done = int(df_new['file_id'].nunique())
                n_suspect_participants = int(df_new.groupby('file_id')['exclude'].any().sum())
                n_suspect = int(df_new['exclude'].sum())
            else:
                n_files_done = n_suspect_participants = n_suspect = 0
            avg_str = f'{elapsed / n_files_done:.0f} s / participant' if n_files_done > 0 else 'N/A'
            print(f'\n=== Analysis complete ===')
            print(f'Output folder              : {reports_root}')
            if (reports_root / 'dataset_overview.html').exists():
                print(f'Dataset overview           : dataset_overview.html')
            print(f'Participants processed     : {n_files_done}')
            print(f'Participants with ≥1 flag  : {n_suspect_participants} / {n_files_done}')
            print(f'Total flagged channels     : {n_suspect}')
            if failed:
                print(f'Files failed to load       : {len(failed)}')
            print(f'Total time                 : {elapsed_str}')
            print(f'Average per participant    : {avg_str}')

    except Exception as _run_err:
        with out:
            print(f'\nUnexpected error — analysis interrupted: {_run_err}')
    finally:
        btn.disabled = False


btn_run.on_click(run_analysis)

display(
    HTML('<h3 style="margin-top:24px;">&#9654; Run Quality Overview</h3>'),
    btn_run,
    widgets.HBox([progress, progress_label]),
    widgets.HBox([widgets.VBox([progress_ch, phase_legend], layout=widgets.Layout(margin='0')),
                  progress_ch_label], layout=widgets.Layout(align_items='center')),
    out
)